# 03 | Sentimientos y alcance de videos de YouTube
### Laboratorio práctico de Procesamiento de Lenguaje Natural
**Pregunta:** ¿cómo se relacionan las visualizaciones de un video con las etiquetas de sus comentarios y qué vocabulario aparece en ellos?

**8 celdas de código · CPU · 2 CSV · sin claves de API.** Este cuaderno funciona de manera independiente: no es necesario ejecutar los books 1 y 2.

Cargue `comments.csv` y `videos-stats.csv` en la celda 01. Ejecute en orden. El código está visible y se entrega sin ejecutar.

**Qué haremos:** limpiar → resumir sentimientos por video → unir con visualizaciones → comparar → normalizar texto → TF-IDF → interpretar → exportar PDF y evidencias.

**Antes de empezar:**
- En este laboratorio, “alcance” significa **visualizaciones registradas (`Views`)**, no personas únicas alcanzadas.
- Se reutiliza `Sentiment` con la convención revisable **0 = Negativo, 1 = Neutral, 2 = Positivo**. No se entrena un nuevo clasificador ni se asignan etiquetas nuevas con TF-IDF.
- El PLN se aplica a la normalización, tokenización y representación de comentarios con **TF-IDF**. Un peso TF-IDF no es una puntuación de sentimiento.
- Los comentarios son una muestra histórica de selección desconocida. Ni sus etiquetas ni las asociaciones prueban satisfacción de toda la audiencia o causalidad.

## Celda 01. Cargar los dos archivos
Escriba su nombre y grupo. Si los archivos aún no están en Colab, selecciónelos juntos cuando aparezca el botón de carga, conservando sus nombres originales.
**Observe:** aquí se leen los archivos, todavía no se calcula la relación entre sentimientos y visualizaciones.

In [ ]:
import re, html, json, shutil, textwrap
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from matplotlib.backends.backend_pdf import PdfPages
from scipy.stats import spearmanr
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from IPython.display import display, FileLink

ESTUDIANTE = "Escriba su nombre"
GRUPO = "Escriba su grupo"
SALIDA = Path("resultados_sentimiento_alcance")
SALIDA.mkdir(exist_ok=True)
figuras = {}
archivos = ["comments.csv", "videos-stats.csv"]
if not all(Path(nombre).is_file() for nombre in archivos):
    try:
        from google.colab import files
    except ImportError as error:
        raise FileNotFoundError("Coloque los dos CSV junto al cuaderno.") from error
    print("Seleccione comments.csv y videos-stats.csv.")
    files.upload()
if not all(Path(nombre).is_file() for nombre in archivos):
    raise FileNotFoundError("Falta un CSV: conserve los nombres originales y repita esta celda.")
comentarios_originales = pd.read_csv(archivos[0], encoding="utf-8-sig")
videos_originales = pd.read_csv(archivos[1], encoding="utf-8-sig")
for tabla, columnas in [(comentarios_originales, ["Video ID", "Comment", "Likes", "Sentiment"]),
                        (videos_originales, ["Video ID", "Title", "Views"])]:
    if not set(columnas).issubset(tabla.columns):
        raise ValueError("El CSV no tiene las columnas esperadas. Revise cuál cargó.")
print("Filas de comentarios:", len(comentarios_originales), "| Filas de videos:", len(videos_originales))
display(comentarios_originales.head(2), videos_originales.head(2))

## Celda 02. Limpiar sin alterar los originales
Se retira el índice de exportación y los duplicados exactos. Se excluyen comentarios vacíos, sin identificador o con etiquetas distintas de 0, 1 y 2.

Para cada video se conserva **la primera fila encontrada**, no necesariamente la más reciente. No se suman filas repetidas. Se detectan diferencias en título o vistas para advertir esta limitación. Las visualizaciones no numéricas, negativas o iguales a cero se excluyen del análisis posterior, que usará escala logarítmica; un cero no se declara error del archivo.

Los comentarios iguales que no sean duplicados exactos permanecen: no conocemos a sus autores. Las posibles inconsistencias de etiquetas requieren revisión humana.

In [ ]:
c = comentarios_originales.loc[:, ~comentarios_originales.columns.str.startswith("Unnamed")].copy()
n_duplicados = int(c.duplicated().sum())
c = c.drop_duplicates().dropna(subset=["Video ID", "Comment"])
c["Video ID"] = c["Video ID"].astype(str).str.strip()
c["Sentiment"] = pd.to_numeric(c["Sentiment"], errors="coerce")
validos = c["Video ID"].ne("") & c["Comment"].astype(str).str.strip().ne("") & c["Sentiment"].isin([0, 1, 2])
c = c.loc[validos].copy()
c["Sentiment"] = c["Sentiment"].astype(int)
ETIQUETAS = {0: "Negativo", 1: "Neutral", 2: "Positivo"}
c["sentimiento"] = c["Sentiment"].map(ETIQUETAS)

v = videos_originales.dropna(subset=["Video ID"]).copy()
v["Video ID"] = v["Video ID"].astype(str).str.strip()
v = v[v["Video ID"].ne("")]
repetidos = v[v["Video ID"].duplicated(keep=False)].copy()
conflictos = v.groupby("Video ID")[["Title", "Views"]].nunique(dropna=False).gt(1).any(axis=1)
v = v.drop_duplicates("Video ID", keep="first")[["Video ID", "Title", "Views"]].copy()
v["Views"] = pd.to_numeric(v["Views"], errors="coerce")
v.loc[~np.isfinite(v["Views"]) | v["Views"].lt(0), "Views"] = np.nan
if c.empty or v.empty:
    raise ValueError("No hay registros válidos para continuar.")
auditoria = pd.DataFrame({"Control": ["Comentarios originales", "Duplicados exactos retirados",
    "Otros comentarios excluidos", "Comentarios válidos", "Filas originales de videos",
    "Videos únicos", "Videos con diferencias de título o vistas", "Videos sin vistas > 0",
    "Comentarios sin video asociado"], "Cantidad": [len(comentarios_originales), n_duplicados,
    len(comentarios_originales)-n_duplicados-len(c), len(c), len(videos_originales), len(v),
    int(conflictos.sum()), int((~v["Views"].gt(0)).sum()), int((~c["Video ID"].isin(v["Video ID"])).sum())]})
display(auditoria)
print("Los originales permanecen intactos. Las etiquetas del CSV no se verificaron manualmente.")

## Celda 03. Resumir sentimientos y relacionarlos con el video
Primero se cuenta cada etiqueta por `Video ID`; luego se une el resumen con los videos únicos. Así, las vistas **no se repiten por cada comentario**.

**Porcentaje positivo = comentarios con etiqueta 2 / comentarios válidos del video × 100.** No se divide entre `Comments` del archivo de videos ni se usa el promedio de los códigos 0, 1 y 2.

Se exige un mínimo didáctico de **5 comentarios disponibles**; no es una garantía de representatividad. Con 10 comentarios, una sola etiqueta cambia el porcentaje en 10 puntos.

Los grupos **Bajo, Medio y Alto** se construyen con los terciles de visualizaciones de la muestra analizada. Son relativos a esta muestra, no categorías oficiales. En el resumen, el promedio de porcentajes da **el mismo peso a cada video**.

In [ ]:
MIN_COMENTARIOS = 5  # Umbral didáctico; después puede probar con 10.
conteos = pd.crosstab(c["Video ID"], c["Sentiment"]).reindex(columns=[0, 1, 2], fill_value=0)
conteos.columns = ["negativos", "neutrales", "positivos"]
conteos["n_comentarios"] = conteos.sum(axis=1)
conteos["pct_positivo"] = 100 * conteos["positivos"] / conteos["n_comentarios"]
conteos["pct_negativo"] = 100 * conteos["negativos"] / conteos["n_comentarios"]
conteos["pct_neutral"] = 100 * conteos["neutrales"] / conteos["n_comentarios"]
integrada = v.merge(conteos.reset_index(), on="Video ID", how="left", validate="one_to_one")
analisis = integrada[integrada["Views"].gt(0) & integrada["n_comentarios"].ge(MIN_COMENTARIOS)].copy()
if len(analisis) < 6 or analisis["Views"].nunique() < 3:
    raise ValueError("Faltan videos con vistas positivas. Revise el umbral y los datos.")
corte1, corte2 = analisis["Views"].quantile([1/3, 2/3])
if corte1 >= corte2:
    raise ValueError("Los valores están muy empatados: no se pueden construir tres grupos útiles.")
analisis["alcance"] = pd.cut(analisis["Views"], [-np.inf, corte1, corte2, np.inf], labels=["Bajo", "Medio", "Alto"])
resumen = analisis.groupby("alcance", observed=True).agg(
    videos=("Video ID", "size"), comentarios=("n_comentarios", "sum"),
    vistas_min=("Views", "min"), vistas_max=("Views", "max"), mediana_vistas=("Views", "median"),
    promedio_pct_positivo=("pct_positivo", "mean"), promedio_pct_negativo=("pct_negativo", "mean")
).reset_index()
detalle = c.merge(analisis[["Video ID", "Title", "Views", "alcance"]],
                  on="Video ID", how="inner", validate="many_to_one").reset_index(drop=True)
historial = []  # Se reinicia si cambia la base de análisis.
print("Videos analizados:", len(analisis), "| Comentarios asociados:", len(detalle))
print("Videos fuera del análisis:", len(v)-len(analisis), "| Comentarios fuera:", len(c)-len(detalle))
print(f"Cortes de visualizaciones: {corte1:,.0f} y {corte2:,.0f} (presentación redondeada).")
display(resumen.round(2))

## Celda 04. ¿Se relacionan las vistas con el sentimiento?
Cada punto es un video. El eje horizontal es logarítmico para poder mostrar videos con magnitudes muy diferentes. **No muestra crecimiento temporal.**

Spearman (`rho`) describe asociación monotónica: valores cercanos a 0 indican poca asociación de ese tipo; el signo indica la dirección. No es una precisión del modelo ni demuestra causalidad. No se realizan pruebas de significancia ni se calculan intervalos de confianza.

In [ ]:
def grafico(titulo, subtitulo, alto=4.8):
    fig, ax = plt.subplots(figsize=(10, alto))
    fig.suptitle(titulo, x=0.10, ha="left", fontsize=16, fontweight="bold")
    ax.set_title(subtitulo, loc="left", fontsize=10, pad=13)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=0.16)
    ax.set_axisbelow(True)
    return fig, ax

rho = np.nan
if analisis["pct_positivo"].nunique() > 1:
    rho = float(spearmanr(analisis["Views"], analisis["pct_positivo"]).statistic)
fig, ax = grafico("Visualizaciones y comentarios con etiqueta positiva",
    f"{len(analisis):,} videos | Un punto = un video | Spearman rho = {rho:.3f}")
ax.scatter(analisis["Views"], analisis["pct_positivo"], s=20, alpha=0.35, edgecolors="none")
ax.set_xscale("log")
ax.set_xlabel("Visualizaciones registradas (escala logarítmica)")
ax.set_ylabel("Comentarios positivos / comentarios analizados")
ax.set_ylim(-3, 103)
ax.yaxis.set_major_formatter(PercentFormatter(100))
fig.text(0.10, 0.01, "Muestra disponible: asociación no equivale a causa ni a satisfacción de toda la audiencia.", fontsize=9)
fig.tight_layout(rect=[0, 0.04, 1, 0.94])
figuras["01_relacion"] = fig
plt.show()
plt.close(fig)

## Celda 05. Comparar los tres niveles de visualizaciones
El gráfico muestra el **promedio de porcentajes positivos calculados por video**, no el porcentaje obtenido juntando todos los comentarios del grupo. El tamaño `n` corresponde a videos.

**Actividad:** identifique qué grupo tiene el promedio mayor y calcule la diferencia Alto − Bajo en **puntos porcentuales**. Una diferencia no demuestra un efecto causal ni una diferencia estadísticamente significativa.

In [ ]:
fig, ax = grafico("Comentarios positivos según nivel de visualizaciones",
    "Promedio de porcentajes por video | Cada video tiene el mismo peso")
valores = resumen["promedio_pct_positivo"].to_numpy()
etiquetas = [f"{fila.alcance} (n={fila.videos})" for fila in resumen.itertuples()]
ax.barh(etiquetas, valores, height=0.55)
ax.invert_yaxis()
ax.set_xlim(0, 112)
ax.set_xticks(np.arange(0, 101, 20))
ax.xaxis.set_major_formatter(PercentFormatter(100))
ax.set_xlabel("Promedio del porcentaje de comentarios positivos por video")
ax.grid(axis="y", visible=False)
for i, valor in enumerate(valores):
    ax.text(valor+1, i, f"{valor:.1f}%", va="center", fontsize=12)
diferencia = float(resumen.set_index("alcance").loc["Alto", "promedio_pct_positivo"] -
                   resumen.set_index("alcance").loc["Bajo", "promedio_pct_positivo"])
fig.text(0.10, 0.01, f"Diferencia Alto - Bajo: {diferencia:+.2f} puntos porcentuales. Grupos relativos a esta muestra.", fontsize=9)
fig.tight_layout(rect=[0, 0.04, 1, 0.94])
figuras["02_niveles"] = fig
plt.show()
plt.close(fig)

## Celda 06. Aplicar PLN: del comentario a un vector TF-IDF
Se conserva `Comment` y se crea `texto_limpio`: minúsculas, entidades HTML decodificadas, enlaces retirados y espacios normalizados. Se expanden algunas contracciones inglesas para conservar la negación.

La tokenización y TF-IDF usan el mismo patrón de palabras. Se filtran palabras vacías de **inglés**, conservando negaciones como `not` y `no`. No se traducen los comentarios ni se aplica lematización.

TF-IDF asigna pesos que combinan presencia de un término y frecuencia documental. La matriz tiene **una fila por comentario y una columna por término**. Se ajusta una sola vez con todos los comentarios de la base analizada; así los filtros siguientes comparten la misma representación.

Este es un análisis exploratorio de vocabulario: **TF-IDF no predice `Sentiment`** y no recibe `Views` como característica. No se evalúa generalización de un clasificador en este cuaderno.

In [ ]:
def normalizar(texto):
    t = html.unescape(str(texto)).lower().replace("’", "'")
    t = re.sub(r"https?://\S+|www\.\S+", " ", t)
    t = re.sub(r"<[^>]+>", " ", t)
    t = re.sub(r"\bwon't\b", "will not", t)
    t = re.sub(r"\bcan't\b", "can not", t)
    t = re.sub(r"n't\b", " not", t)
    return re.sub(r"\s+", " ", t).strip()

PATRON = r"(?u)\b[^\W\d_][\w']*\b"
detalle["texto_limpio"] = detalle["Comment"].map(normalizar)
stop = sorted(set(ENGLISH_STOP_WORDS) - {"no", "not", "nor", "never"})
vectorizador = TfidfVectorizer(lowercase=False, token_pattern=PATRON,
    stop_words=stop, min_df=2, max_features=4000, sublinear_tf=True)
try:
    matriz = vectorizador.fit_transform(detalle["texto_limpio"])
except ValueError as error:
    raise ValueError("No hay vocabulario suficiente: revise el corpus y min_df.") from error
terminos = vectorizador.get_feature_names_out()
muestra_texto = detalle[["Comment", "texto_limpio"]].head(2).copy()
muestra_texto["tokens_tras_filtro"] = muestra_texto["texto_limpio"].map(vectorizador.build_analyzer())
display(muestra_texto)
print("Matriz TF-IDF:", matriz.shape, "(comentarios, términos)")
print("Comentarios sin términos dentro del vocabulario:", int((matriz.getnnz(axis=1)==0).sum()))

## Celda 07. Experimentar con el lenguaje y leer ejemplos
Cambie **solo** `SENTIMIENTO` y `ALCANCE`. Pruebe: **Negativo/Alto**, **Positivo/Alto** y **Negativo/Bajo**. Repita esta celda después de cada cambio; no hace falta recalcular TF-IDF.

Los términos se ordenan por su **peso TF-IDF promedio entre los comentarios seleccionados**. Son términos destacados en ese subconjunto, no necesariamente exclusivos, causas de popularidad ni palabras que definan automáticamente el sentimiento. `Comentarios_con_termino` cuenta presencia por comentario, no repeticiones de la palabra.

Se muestran dos comentarios que contienen el primer término para revisar su contexto. La tabla `historial` conserva el resumen de los filtros probados. **Los filtros no cambian la correlación global ni los gráficos 04 y 05.**

In [ ]:
SENTIMIENTO = "Negativo"  # Negativo, Neutral o Positivo.
ALCANCE = "Alto"         # Bajo, Medio o Alto.
mascara = (detalle["sentimiento"].eq(SENTIMIENTO) & detalle["alcance"].eq(ALCANCE)).to_numpy()
if not mascara.any():
    raise ValueError("No hay comentarios para este filtro. Revise los nombres y cambie la selección.")
submatriz = matriz[mascara]
palabras = pd.DataFrame({"Termino": terminos, "Peso_TFIDF": np.asarray(submatriz.mean(axis=0)).ravel(),
    "Comentarios_con_termino": np.asarray((submatriz>0).sum(axis=0)).ravel()})
palabras = palabras[palabras["Peso_TFIDF"]>0].nlargest(8, "Peso_TFIDF")
if palabras.empty:
    raise ValueError("Este filtro no tiene términos en el vocabulario. Elija otro.")
fig, ax = grafico("Vocabulario destacado en los comentarios", f"{SENTIMIENTO} | Alcance {ALCANCE} | {mascara.sum():,} comentarios")
ax.barh(palabras["Termino"], palabras["Peso_TFIDF"])
ax.invert_yaxis()
ax.set_xlabel("Peso TF-IDF promedio (no es un porcentaje ni una puntuación de sentimiento)")
ax.grid(axis="y", visible=False)
fig.tight_layout(rect=[0, 0, 1, 0.94])
figuras["03_vocabulario"] = fig
plt.show()
plt.close(fig)
display(palabras.round(4))
termino = palabras.iloc[0]["Termino"]
seleccion = detalle.loc[mascara]
ejemplos = seleccion[seleccion["texto_limpio"].str.contains(r"\b"+re.escape(termino)+r"\b", regex=True)].head(2)
with pd.option_context("display.max_colwidth", None):
    display(ejemplos[["Video ID", "Title", "Comment", "sentimiento"]])
registro = {"Sentimiento": SENTIMIENTO, "Alcance": ALCANCE, "Comentarios": int(mascara.sum()),
            "Primeros_terminos": ", ".join(palabras["Termino"].head(3))}
historial = [fila for fila in historial if (fila["Sentimiento"],fila["Alcance"]) != (SENTIMIENTO,ALCANCE)] + [registro]
display(pd.DataFrame(historial))

## Celda 08. Escribir la conclusión y descargar las evidencias
Complete las tres respuestas breves y ejecute. Se genera **un PDF de cuatro páginas** (resumen y tres gráficos) y un ZIP con el PDF, gráficos y tablas CSV. El PDF y el gráfico de vocabulario corresponden al **último filtro**; `experimentos.csv` conserva todos los filtros probados en esta sesión.

El PDF no reemplaza el taller diligenciado. Para guardar otro filtro, repita las celdas **07 y 08**. Si modifica el mínimo de comentarios, repita desde la **03** y vuelva a realizar los experimentos. Los CSV originales no se sobrescriben.

In [ ]:
CONCLUSION = "Complete: qué relación observó y con qué cifra la sustenta."
HALLAZGO_PLN = "Complete: qué término revisó y qué significaba en su contexto."
LIMITACION = "Complete: una conclusión que estos datos no permiten asegurar."
if any(len(t)>300 for t in [CONCLUSION, HALLAZGO_PLN, LIMITACION]):
    raise ValueError("Escriba máximo 300 caracteres por respuesta para mantener legible el informe.")
lineas = [f"Estudiante: {ESTUDIANTE} | Grupo: {GRUPO}",
    f"Videos: {len(analisis)} | Comentarios: {len(detalle)} | Mínimo por video: {MIN_COMENTARIOS}",
    f"Spearman rho: {rho:.3f} | Diferencia Alto - Bajo: {diferencia:+.2f} puntos porcentuales.",
    f"Filtro de vocabulario: {SENTIMIENTO} / {ALCANCE} | Comentarios: {int(mascara.sum())}",
    "Conclusión: " + CONCLUSION, "Hallazgo de PLN: " + HALLAZGO_PLN, "Limitación: " + LIMITACION,
    "Lectura: se usan etiquetas existentes, no nuevas predicciones. TF-IDF describe vocabulario, no sentimiento.",
    "Límites: muestra histórica de selección desconocida; visualizaciones no son personas únicas; asociación no es causalidad.",
    "Trazabilidad: los CSV adjuntos contienen conteos, porcentajes, auditoría y filtros. Consulte las referencias técnicas del cuaderno."]
with PdfPages(SALIDA / "informe_sentimiento_alcance.pdf") as pdf:
    portada = plt.figure(figsize=(8.27, 11.69))
    portada.text(0.08, 0.94, "SENTIMIENTOS Y ALCANCE EN YOUTUBE", fontsize=15, fontweight="bold")
    portada.text(0.08, 0.89, "\n\n".join(textwrap.fill(t, 85) for t in lineas), va="top", fontsize=11, linespacing=1.5)
    pdf.savefig(portada)
    plt.close(portada)
    for nombre, figura in figuras.items():
        pdf.savefig(figura, bbox_inches="tight")
        figura.savefig(SALIDA / f"{nombre}.png", dpi=180, bbox_inches="tight")
tablas = {"videos_analizados": analisis, "resumen_alcance": resumen, "auditoria": auditoria,
    "vocabulario_ultimo_filtro": palabras, "ejemplos_ultimo_filtro": ejemplos,
    "experimentos": pd.DataFrame(historial), "videos_repetidos_originales": repetidos}
for nombre, tabla in tablas.items():
    tabla.to_csv(SALIDA / f"{nombre}.csv", index=False, encoding="utf-8-sig")
(SALIDA / "configuracion.json").write_text(json.dumps({"min_comentarios": MIN_COMENTARIOS,
    "cortes_vistas": [float(corte1), float(corte2)], "sentimiento": SENTIMIENTO, "alcance": ALCANCE,
    "rho": None if np.isnan(rho) else rho, "TFIDF": "min_df=2; max_features=4000; unigramas; stopwords inglés"}, ensure_ascii=False, indent=2), encoding="utf-8")
paquete = shutil.make_archive(str(SALIDA), "zip", root_dir=SALIDA)
display(FileLink(str(SALIDA / "informe_sentimiento_alcance.pdf")), FileLink(paquete))
try:
    from google.colab import files
    files.download(paquete)  # El ZIP contiene el PDF y todas las evidencias.
except ImportError:
    print("Abra los enlaces anteriores. Fuera de Colab, los resultados están en:", SALIDA.resolve())

### Cierre y fuentes
**Entregue:** el taller diligenciado, su copia del cuaderno ejecutado y el ZIP generado. No basta con pegar gráficos: cite cifras, lea ejemplos y explique límites.

**Fuentes de datos:** `comments.csv` y `videos-stats.csv` suministrados para la clase. No se conoce la fecha de captura ni el método de selección de comentarios. No se consultan ni actualizan datos en YouTube.

**Documentación técnica:**
- pandas, uniones y validación de claves: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html
- scikit-learn, TF-IDF: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html
- SciPy, correlación de Spearman: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.spearmanr.html

**Solución de problemas:** cargue los dos CSV con sus nombres originales; ejecute en orden; ante un filtro vacío seleccione otra combinación; si Colab se reinicia, vuelva a cargar los archivos y ejecute desde la celda 01. Al comparar resultados, mantenga iguales el mínimo de comentarios y la base utilizada.